# IoT Wearable Function Name Predictor — Hierarchical Edge AI Pipeline

**Objective:** Predict camelCase function names for an embedded IoT/fitness wearable device SDK from natural-language descriptions and keywords.

**Constraints:**
- Model size < 5 MB (embedded chip deployment)
- Sub-millisecond inference on low-power CPU
- 100% offline — no cloud APIs, no internet
- Must handle semantic variance in user commands

**Architecture (4 layers):**
1. **Flat Baseline** — TF-IDF + LogisticRegression (word + character n-grams)
2. **Hierarchical Two-Stage Router** — Intent classification → Target classification
3. **Character N-gram Fusion** — Sub-word features for typo/variant robustness
4. **OOD Reject Gate** — Entropy-based out-of-distribution detection for safety

**Dataset:** 70 unique IoT/fitness functions across 6 intent groups, augmented to 210 samples.

In [1]:
import re
import time
import os
import warnings
import unicodedata
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack
from scipy.stats import entropy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

print('All imports loaded successfully.')

All imports loaded successfully.


## Section 1 — Dataset Loading, Cleaning & Augmentation

The new dataset has **70 unique functions** for an IoT fitness wearable SDK. Key challenges:
- Only `description` and `keywords` columns contain data — `parameters`, `return_type`, `library` are all empty
- Only **1 sample per function** — requires augmentation to enable train/test splitting
- Descriptions are heavily **templated** within intent groups (differ by one word)
- **2 rows** contain corrupted Unicode characters that need cleaning

In [2]:
# ── Load and Clean Dataset ───────────────────────────────────────────────────

df_raw = pd.read_csv('improved_functions_1.csv')
print(f'Raw dataset: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'Columns: {list(df_raw.columns)}')
print(f'Unique functions: {df_raw["function_name"].nunique()}')

# Drop empty columns (parameters, return_type, library are all NaN)
empty_cols = [c for c in df_raw.columns if df_raw[c].isna().all()]
print(f'\nEmpty columns (dropping): {empty_cols}')
df_raw.drop(columns=empty_cols, inplace=True)

# ── Unicode Cleaning ─────────────────────────────────────────────────────────
def clean_unicode(text):
    """Remove mojibake and non-ASCII characters from text.
    The dataset has corrupted UTF-8 sequences in the Music rows.
    Only keep printable ASCII characters (0x20-0x7E) and standard whitespace."""
    cleaned = ''.join(ch for ch in str(text) if 32 <= ord(ch) <= 126)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

df_raw['description'] = df_raw['description'].apply(clean_unicode)
df_raw['keywords']    = df_raw['keywords'].apply(clean_unicode)

# Verify Music rows are cleaned
music_rows = df_raw[df_raw['function_name'].str.contains('Music')]
print(f'\nCleaned Music rows:')
for _, row in music_rows.iterrows():
    print(f'  {row["function_name"]}: "{row["description"][:60]}..."')
    print(f'    keywords: {row["keywords"]}')

print(f'\nFinal cleaned shape: {df_raw.shape}')
df_raw.head(10)

Raw dataset: 70 rows x 6 columns
Columns: ['description', 'parameters', 'return_type', 'library', 'keywords', 'function_name']
Unique functions: 70

Empty columns (dropping): ['parameters', 'return_type', 'library']

Cleaned Music rows:
  enableMusicControl: "Enables remote control functionality for Music operations...."
    keywords: enable, switch on, turn off, music, remote control
  disableMusicControl: "Disables remote control functionality for Music operations...."
    keywords: disable, switch on, turn off, music, remote control

Final cleaned shape: (70, 3)


,description,keywords,function_name
0,"Begins recording a Running workout session, tr...","track, record, begin, start, running, exercise...",startRunning
1,Ends the current Running session and saves the...,"finish, end, stop, running, complete, save",stopRunning
2,"Begins recording a Walking workout session, tr...","track, record, begin, start, walking, exercise...",startWalking
3,Ends the current Walking session and saves the...,"finish, end, stop, walking, complete, save",stopWalking
4,"Begins recording a Yoga workout session, track...","track, record, begin, start, yoga, exercise, w...",startYoga
5,Ends the current Yoga session and saves the fi...,"finish, end, stop, yoga, complete, save",stopYoga
6,"Begins recording a Swimming workout session, t...","track, record, begin, start, swimming, exercis...",startSwimming
7,Ends the current Swimming session and saves th...,"finish, end, stop, swimming, complete, save",stopSwimming
8,Enables remote control functionality for Music...,"enable, switch on, turn off, music, remote con...",enableMusicControl
9,Disables remote control functionality for Musi...,"disable, switch on, turn off, music, remote co...",disableMusicControl


In [3]:
# ── Data Augmentation ────────────────────────────────────────────────────────
# With only 1 sample per function, we cannot do train/test splits.
# Generate 2 additional description variants per function (total: 210 rows).

def camel_tokenize(text):
    """Split CamelCase into lowercase words: MathUtils -> math utils"""
    return re.sub(r'([A-Z])', r' \1', str(text)).lower().strip()

def extract_activity(func_name):
    """Extract the activity/entity name from a camelCase function name."""
    for prefix in ['start', 'stop', 'live', 'enable', 'disable']:
        if func_name.startswith(prefix) and func_name != prefix:
            entity = func_name[len(prefix):]
            entity = entity.replace('Control', '')
            return entity
    return func_name

def assign_intent(func_name):
    """Assign intent group from function name prefix."""
    if func_name.startswith('start'):  return 'start_workout'
    if func_name.startswith('stop'):   return 'stop_workout'
    if func_name.startswith('live'):   return 'live_monitor'
    if func_name.startswith('enable'): return 'enable_control'
    if func_name.startswith('disable'):return 'disable_control'
    return 'get_data'

# Augmentation templates: {activity} is replaced with the tokenized activity name
AUG_TEMPLATES = {
    'start_workout': [
        "Initiates a {activity} exercise session and starts tracking workout metrics.",
        "Launches the {activity} fitness activity to monitor performance data."
    ],
    'stop_workout': [
        "Finishes the {activity} workout and stores the recorded fitness metrics.",
        "Completes the active {activity} session and saves all tracked data."
    ],
    'live_monitor': [
        "Provides live {activity} readings from the wearable sensors in real time.",
        "Continuously monitors {activity} data from device sensors."
    ],
    'get_data': [
        "Fetches stored {activity} data including history and summary reports.",
        "Returns the user's {activity} statistics and historical records."
    ],
    'enable_control': [
        "Activates the remote {activity} control feature on the connected device.",
        "Turns on remote access capability for {activity} functions."
    ],
    'disable_control': [
        "Deactivates the remote {activity} control feature on the connected device.",
        "Turns off remote access capability for {activity} functions."
    ]
}

# Build augmented dataset
aug_rows = []
rng = np.random.RandomState(42)

for idx, row in df_raw.iterrows():
    fname   = row['function_name']
    intent  = assign_intent(fname)
    entity  = extract_activity(fname)
    activity_text = camel_tokenize(entity)

    # Original row
    aug_rows.append({
        'description':   row['description'],
        'keywords':      row['keywords'],
        'function_name': fname,
        'intent':        intent,
        'variant':       'original'
    })

    # 2 augmented variants
    templates = AUG_TEMPLATES.get(intent, AUG_TEMPLATES['get_data'])
    for i, tmpl in enumerate(templates):
        # Shuffle keyword order for TF-IDF variation
        kw_list = [k.strip() for k in row['keywords'].split(',')]
        rng.shuffle(kw_list)
        shuffled_kw = ', '.join(kw_list)

        aug_rows.append({
            'description':   tmpl.format(activity=activity_text),
            'keywords':      shuffled_kw,
            'function_name': fname,
            'intent':        intent,
            'variant':       f'variant_{i+1}'
        })

df = pd.DataFrame(aug_rows)

print(f'Augmented dataset: {len(df)} rows ({df["function_name"].nunique()} unique functions)')
print(f'Variants per function: {df.groupby("function_name").size().unique()}')
print(f'\nIntent distribution:')
print(df['intent'].value_counts().to_string())

# Show sample augmentation for one function
print(f'\n── Sample augmentation (startYoga) ─────────────────────────')
for _, r in df[df['function_name'] == 'startYoga'].iterrows():
    print(f'  [{r["variant"]}] {r["description"]}')
    print(f'    keywords: {r["keywords"]}')

Augmented dataset: 210 rows (70 unique functions)
Variants per function: [3]

Intent distribution:
intent
start_workout      72
stop_workout       72
get_data           24
enable_control     15
disable_control    15
live_monitor       12

── Sample augmentation (startYoga) ─────────────────────────
  [original] Begins recording a Yoga workout session, tracking metrics like duration and calories.
    keywords: track, record, begin, start, yoga, exercise, workout
  [variant_1] Initiates a yoga exercise session and starts tracking workout metrics.
    keywords: record, workout, start, exercise, yoga, begin, track
  [variant_2] Launches the yoga fitness activity to monitor performance data.
    keywords: yoga, begin, exercise, workout, start, record, track


## Section 2 — Feature Engineering & Vectorization

**Key changes from original pipeline:**
- `build_feature_text()` now uses only `description` + `keywords` (no params, return_type, library, param_count)
- Keywords are still repeated **3x** for TF-IDF signal boosting — even more critical now since keywords contain the primary discriminative token (the sport/activity name)
- Two vectorizers: **word-level TF-IDF** (semantic) + **character n-gram TF-IDF** (sub-word robustness)
- Features are concatenated via `scipy.sparse.hstack`

In [4]:
# ── Feature Engineering ───────────────────────────────────────────────────────

def build_feature_text(row):
    """
    Combine description + keywords into enriched feature string.
    Keywords repeated 3x for stronger TF-IDF signal (they contain the
    primary discriminative token — the activity/sport name).
    """
    desc = str(row['description']).lower()
    kw   = str(row['keywords']).replace(',', ' ').lower()
    return f"{desc} {kw} {kw} {kw}"

df['feature_text'] = df.apply(build_feature_text, axis=1)

# ── Label Encoding ───────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df['function_name'])
print(f'Function classes: {len(le.classes_)} unique function names')

intent_le = LabelEncoder()
y_intent  = intent_le.fit_transform(df['intent'])
print(f'Intent classes:   {len(intent_le.classes_)} → {list(intent_le.classes_)}')

# ── TF-IDF Vectorization (Word + Character N-gram Fusion) ────────────────────

# Word-level: captures semantic tokens like "running", "heart rate", "remote control"
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=2000,
    sublinear_tf=True,
    min_df=1
)
X_word = word_vectorizer.fit_transform(df['feature_text'])

# Character-level: captures sub-word patterns for typo/variant robustness
# "swimming" and "swiming" share trigrams: "swi", "wim", "min", "ing"
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),
    max_features=1000,
    sublinear_tf=True,
    min_df=1
)
X_char = char_vectorizer.fit_transform(df['feature_text'])

# Combined feature matrix
X_combined = hstack([X_word, X_char])

print(f'\nWord features  : {X_word.shape[1]} dimensions')
print(f'Char features  : {X_char.shape[1]} dimensions')
print(f'Combined       : {X_combined.shape[0]} rows x {X_combined.shape[1]} features')
print(f'Sparsity       : {100*(1 - X_combined.nnz/(X_combined.shape[0]*X_combined.shape[1])):.1f}%')

# ── Stratified Train/Test Split ──────────────────────────────────────────────
# 210 rows, 70 classes, 3 per class → 2 train + 1 test per class
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_combined, y, np.arange(len(df)),
    test_size=70, random_state=42, stratify=y
)

# Also split word-only and intent labels using the same indices
X_word_train = X_word[idx_train]
X_word_test  = X_word[idx_test]
intent_train = df['intent'].values[idx_train]
intent_test  = df['intent'].values[idx_test]
texts_train  = df['feature_text'].values[idx_train]
texts_test   = df['feature_text'].values[idx_test]

print(f'\nTrain: {X_train.shape[0]} samples ({X_train.shape[0]//len(le.classes_)} per class)')
print(f'Test:  {X_test.shape[0]} samples (1 per class)')

Function classes: 70 unique function names
Intent classes:   6 → ['disable_control', 'enable_control', 'get_data', 'live_monitor', 'start_workout', 'stop_workout']

Word features  : 1035 dimensions
Char features  : 1000 dimensions
Combined       : 210 rows x 2035 features
Sparsity       : 89.6%

Train: 140 samples (2 per class)
Test:  70 samples (1 per class)


## Section 3 — Flat Baseline (Word-only vs Word+Char Ablation)

Train the same models as the original pipeline (LogisticRegression + LinearSVC) to establish baselines. Compare word-only TF-IDF vs word+char fused TF-IDF to measure the impact of character n-gram fusion.

In [5]:
# ── Flat Baseline: Train & Evaluate ──────────────────────────────────────────

def evaluate_model(model, X_tr, y_tr, X_te, y_te, X_full, y_full, name, lr_ref=None):
    """Train, evaluate, and return metrics for a single model."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_ms = (time.time() - t0) * 1000

    y_pred   = model.predict(X_te)
    test_acc = accuracy_score(y_te, y_pred)

    cv = ShuffleSplit(n_splits=5, test_size=70, random_state=42)
    cv_scores = cross_val_score(model, X_full, y_full, cv=cv, scoring='accuracy')

    # Inference latency
    t1 = time.time()
    for _ in range(500):
        model.predict(X_te[0])
    infer_ms = (time.time() - t1) / 500 * 1000

    # Top-3 accuracy (only for models with predict_proba)
    top3_acc = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)
        top3_acc = sum(
            1 for i, true in enumerate(y_te)
            if true in np.argsort(proba[i])[::-1][:3]
        ) / len(y_te)

    return {
        'model': model, 'test_acc': test_acc, 'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(), 'top3_acc': top3_acc,
        'train_ms': train_ms, 'infer_ms': infer_ms
    }

# ── Train all flat baselines ─────────────────────────────────────────────────
configs = [
    ('LR (word only)',     LogisticRegression(C=2.0, max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42), X_word_train, X_word_test, X_word),
    ('LR (word+char)',     LogisticRegression(C=2.0, max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42), X_train, X_test, X_combined),
    ('LinearSVC (word+char)', LinearSVC(C=1.5, max_iter=2000, class_weight='balanced', random_state=42), X_train, X_test, X_combined),
]

flat_results = {}
print(f'{"Model":<25} {"Test Acc":>10} {"CV Mean":>10} {"CV Std":>8} {"Top-3":>8} {"Train ms":>10} {"Infer ms":>10}')
print('─' * 90)

for name, model, Xtr, Xte, Xfull in configs:
    r = evaluate_model(model, Xtr, y_train, Xte, y_test, Xfull, y, name)
    flat_results[name] = r
    top3_str = f'{r["top3_acc"]:.4f}' if r['top3_acc'] is not None else '   N/A'
    print(f'{name:<25} {r["test_acc"]:>10.4f} {r["cv_mean"]:>10.4f} {r["cv_std"]:>8.4f} {top3_str:>8} {r["train_ms"]:>10.1f} {r["infer_ms"]:>10.4f}')

# Select best flat model
best_flat_name = max(
    [k for k in flat_results if 'LR' in k],
    key=lambda k: flat_results[k]['test_acc']
)
best_flat = flat_results[best_flat_name]
print(f'\nBest flat model: {best_flat_name} (test acc: {best_flat["test_acc"]:.4f})')

Model                       Test Acc    CV Mean   CV Std    Top-3   Train ms   Infer ms
──────────────────────────────────────────────────────────────────────────────────────────


LR (word only)                0.7000     0.4371   0.0475   0.9000       21.4     0.0794


LR (word+char)                0.9857     0.7714   0.0670   1.0000       54.6     0.1063


LinearSVC (word+char)         0.9714     0.7371   0.0265      N/A       88.6     0.0513

Best flat model: LR (word+char) (test acc: 0.9857)


## Section 4 — Hierarchical Two-Stage Intent Router

**Architecture:**
- **Stage 1 (Intent Classifier):** 6-class LogisticRegression classifies the user's intent (start_workout, stop_workout, live_monitor, get_data, enable_control, disable_control)
- **Stage 2 (Target Classifiers):** One sub-classifier per intent group, trained only on that group's data. Each gets its own TF-IDF vectorizer for accurate IDF weighting within the group.

**Why this helps:** A flat 70-class classifier sees 24 nearly-identical `start_*` descriptions and struggles to distinguish them. The hierarchical router first solves the easy problem (which intent?) then solves the hard problem in a smaller, focused space (which sport?).

In [6]:
# ── Hierarchical Two-Stage Predictor ─────────────────────────────────────────

class HierarchicalPredictor:
    """
    Two-stage classifier: Intent → Target function.
    Stage 1: Classify into 6 intent groups using combined word+char TF-IDF.
    Stage 2: Per-intent sub-classifiers with group-local TF-IDF vectorizers.
    """

    def __init__(self):
        # Stage 1 components
        self.s1_word_vec  = None
        self.s1_char_vec  = None
        self.s1_model     = None
        self.s1_le        = None   # intent label encoder

        # Stage 2 components: intent_name -> (word_vec, char_vec, model)
        self.s2_models    = {}
        self.func_le      = None   # function name label encoder

    def fit(self, texts, y_func, intents, func_le, intent_le):
        """
        texts:     array of raw feature_text strings
        y_func:    array of encoded function name labels
        intents:   array of intent strings (e.g., 'start_workout')
        func_le:   fitted LabelEncoder for function names
        intent_le: fitted LabelEncoder for intents
        """
        self.func_le = func_le
        self.s1_le   = intent_le

        # ── Stage 1: Intent Classifier ───────────────────────────────────
        y_intent = intent_le.transform(intents)

        self.s1_word_vec = TfidfVectorizer(ngram_range=(1,2), max_features=2000,
                                            sublinear_tf=True, min_df=1)
        self.s1_char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,6),
                                            max_features=1000, sublinear_tf=True, min_df=1)

        X_s1_word = self.s1_word_vec.fit_transform(texts)
        X_s1_char = self.s1_char_vec.fit_transform(texts)
        X_s1 = hstack([X_s1_word, X_s1_char])

        self.s1_model = LogisticRegression(C=2.0, max_iter=1000, solver='lbfgs',
                                           class_weight='balanced', random_state=42)
        self.s1_model.fit(X_s1, y_intent)

        s1_train_acc = accuracy_score(y_intent, self.s1_model.predict(X_s1))
        print(f'Stage 1 (Intent): {len(intent_le.classes_)} classes, train acc = {s1_train_acc:.4f}')

        # ── Stage 2: Per-intent Sub-classifiers ──────────────────────────
        for intent_name in sorted(set(intents)):
            mask = np.array([i == intent_name for i in intents])
            sub_texts = texts[mask]
            sub_y     = y_func[mask]

            n_classes = len(set(sub_y))
            if n_classes <= 1:
                # Single-class group — no classifier needed, store the label
                self.s2_models[intent_name] = ('single', sub_y[0])
                print(f'  Stage 2 [{intent_name:>16}]: 1 class (direct mapping)')
                continue

            # Group-local TF-IDF (critical: IDF is computed within the group only)
            s2_wv = TfidfVectorizer(ngram_range=(1,2), max_features=1500,
                                    sublinear_tf=True, min_df=1)
            s2_cv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,6),
                                    max_features=800, sublinear_tf=True, min_df=1)

            X_s2 = hstack([s2_wv.fit_transform(sub_texts), s2_cv.fit_transform(sub_texts)])

            s2_model = LogisticRegression(C=2.0, max_iter=1000, solver='lbfgs',
                                          class_weight='balanced', random_state=42)
            s2_model.fit(X_s2, sub_y)

            self.s2_models[intent_name] = ('classifier', s2_wv, s2_cv, s2_model)

            s2_acc = accuracy_score(sub_y, s2_model.predict(X_s2))
            print(f'  Stage 2 [{intent_name:>16}]: {n_classes} classes, train acc = {s2_acc:.4f}')

    def _predict_single(self, text):
        """Predict a single text input. Returns (func_label, intent_proba, func_proba)."""
        # Stage 1: predict intent
        X_s1 = hstack([
            self.s1_word_vec.transform([text]),
            self.s1_char_vec.transform([text])
        ])
        intent_proba = self.s1_model.predict_proba(X_s1)[0]
        intent_idx   = np.argmax(intent_proba)
        intent_name  = self.s1_le.inverse_transform([intent_idx])[0]
        intent_conf  = intent_proba[intent_idx]

        # Stage 2: predict function within intent group
        s2_entry = self.s2_models[intent_name]
        if s2_entry[0] == 'single':
            return s2_entry[1], intent_conf, 1.0, intent_proba

        _, s2_wv, s2_cv, s2_model = s2_entry
        X_s2 = hstack([s2_wv.transform([text]), s2_cv.transform([text])])
        func_proba = s2_model.predict_proba(X_s2)[0]
        func_idx   = np.argmax(func_proba)
        func_label = s2_model.predict(X_s2)[0]
        func_conf  = func_proba[func_idx]

        return func_label, intent_conf, func_conf, func_proba

    def predict(self, texts):
        """Predict function labels for an array of texts."""
        return np.array([self._predict_single(t)[0] for t in texts])

    def predict_with_confidence(self, text, top_k=3):
        """Predict a single text with confidence scores and top-k alternatives."""
        t0 = time.time()
        func_label, intent_conf, func_conf, func_proba = self._predict_single(text)
        latency = (time.time() - t0) * 1000

        func_name = self.func_le.inverse_transform([func_label])[0]

        # Determine intent for proper Stage 2 label decoding
        X_s1 = hstack([self.s1_word_vec.transform([text]), self.s1_char_vec.transform([text])])
        intent_idx  = self.s1_model.predict(X_s1)[0]
        intent_name = self.s1_le.inverse_transform([intent_idx])[0]

        # Build top-k list from Stage 2 probabilities
        s2_entry = self.s2_models[intent_name]
        if s2_entry[0] == 'single':
            top_k_list = [(func_name, 1.0)]
        else:
            _, _, _, s2_model = s2_entry
            top_idxs = np.argsort(func_proba)[::-1][:top_k]
            top_k_list = [
                (self.func_le.inverse_transform([s2_model.classes_[i]])[0],
                 round(float(func_proba[i]), 4))
                for i in top_idxs
            ]

        return {
            'prediction':  func_name,
            'intent':      intent_name,
            'intent_conf': round(float(intent_conf), 4),
            'func_conf':   round(float(func_conf), 4),
            'top_k':       top_k_list,
            'latency_ms':  round(latency, 4)
        }

print('HierarchicalPredictor class defined.')

HierarchicalPredictor class defined.


In [7]:
# ── Train & Evaluate Hierarchical Router ─────────────────────────────────────

hier = HierarchicalPredictor()
hier.fit(texts_train, y_train, intent_train, le, intent_le)

# ── Test set evaluation ──────────────────────────────────────────────────────
y_pred_hier = hier.predict(texts_test)
hier_acc    = accuracy_score(y_test, y_pred_hier)

# Stage 1 accuracy on test set
X_s1_test = hstack([
    hier.s1_word_vec.transform(texts_test),
    hier.s1_char_vec.transform(texts_test)
])
s1_pred_test = hier.s1_model.predict(X_s1_test)
s1_true_test = intent_le.transform(intent_test)
s1_test_acc  = accuracy_score(s1_true_test, s1_pred_test)

# Top-3 accuracy
top3_correct = 0
for text, true_label in zip(texts_test, y_test):
    result = hier.predict_with_confidence(text, top_k=3)
    top3_names = [t[0] for t in result['top_k']]
    true_name  = le.inverse_transform([true_label])[0]
    if true_name in top3_names:
        top3_correct += 1
hier_top3 = top3_correct / len(y_test)

# Inference latency benchmark
latencies = []
for _ in range(200):
    t = time.time()
    hier._predict_single(texts_test[0])
    latencies.append((time.time() - t) * 1000)
lat = np.array(latencies)

print(f'\n{"=" * 65}')
print(f'         HIERARCHICAL ROUTER — TEST RESULTS')
print(f'{"=" * 65}')
print(f'  Stage 1 accuracy (intent):  {s1_test_acc:.4f}')
print(f'  Overall accuracy (func):    {hier_acc:.4f}')
print(f'  Top-3 accuracy:             {hier_top3:.4f}')
print(f'  Inference latency (mean):   {lat.mean():.4f} ms')
print(f'  Inference latency (median): {np.median(lat):.4f} ms')
print(f'  Throughput:                 {1000/lat.mean():.0f} predictions/sec')
print(f'{"=" * 65}')

# ── Comparison: Flat vs Hierarchical ─────────────────────────────────────────
print(f'\n{"Model":<25} {"Test Acc":>10} {"Top-3":>8}')
print('─' * 50)
for name, r in flat_results.items():
    top3_str = f'{r["top3_acc"]:.4f}' if r['top3_acc'] else '   N/A'
    print(f'{name:<25} {r["test_acc"]:>10.4f} {top3_str:>8}')
print(f'{"Hierarchical Router":<25} {hier_acc:>10.4f} {hier_top3:>8.4f}')

Stage 1 (Intent): 6 classes, train acc = 1.0000
  Stage 2 [ disable_control]: 5 classes, train acc = 1.0000
  Stage 2 [  enable_control]: 5 classes, train acc = 1.0000
  Stage 2 [        get_data]: 8 classes, train acc = 1.0000
  Stage 2 [    live_monitor]: 4 classes, train acc = 1.0000
  Stage 2 [   start_workout]: 24 classes, train acc = 1.0000
  Stage 2 [    stop_workout]: 24 classes, train acc = 1.0000



         HIERARCHICAL ROUTER — TEST RESULTS
  Stage 1 accuracy (intent):  1.0000
  Overall accuracy (func):    1.0000
  Top-3 accuracy:             1.0000
  Inference latency (mean):   0.9701 ms
  Inference latency (median): 0.9642 ms
  Throughput:                 1031 predictions/sec

Model                       Test Acc    Top-3
──────────────────────────────────────────────────
LR (word only)                0.7000   0.9000
LR (word+char)                0.9857   1.0000
LinearSVC (word+char)         0.9714      N/A
Hierarchical Router           1.0000   1.0000


## Section 5 — Out-of-Distribution (OOD) Reject Gate

On embedded hardware, executing the wrong function is a **hardware failure**, not just a UX failure. If a user says something the model doesn't recognize, it should **reject** rather than confidently returning a wrong answer.

**Two complementary signals:**
1. **Entropy** of predict_proba: high entropy = model is confused across many classes
2. **Confidence gap**: difference between top-1 and top-2 probabilities. Small gap = ambiguous prediction

In [8]:
# ── OOD Reject Gate ──────────────────────────────────────────────────────────

class ConfidenceGate:
    """
    Safety wrapper that rejects out-of-distribution inputs before execution.
    Uses Shannon entropy + confidence gap as dual rejection signals.
    """

    def __init__(self, predictor, entropy_threshold=None, gap_threshold=None):
        self.predictor = predictor
        self.entropy_threshold = entropy_threshold
        self.gap_threshold     = gap_threshold

    def calibrate(self, texts, y_true):
        """
        Set thresholds from training data predictions.
        Conservative calibration: allow all correct predictions through,
        only reject what's clearly outside the learned distribution.
        """
        entropies = []
        gaps = []

        for text, true_label in zip(texts, y_true):
            result = self.predictor.predict_with_confidence(text, top_k=3)
            pred_name = result['prediction']
            true_name = self.predictor.func_le.inverse_transform([true_label])[0]

            if pred_name == true_name:
                probs = [p for _, p in result['top_k']]
                if len(probs) >= 2:
                    gaps.append(probs[0] - probs[1])
                else:
                    gaps.append(probs[0])
                entropies.append(-sum(p * np.log(p + 1e-10) for p in probs if p > 0))

        # Conservative thresholds: use max entropy observed * 2 and min gap * 0.3
        # This ensures near-zero false rejection on in-distribution data
        # while still catching clearly OOD inputs (which have much higher entropy)
        if entropies:
            self.entropy_threshold = max(entropies) * 2.0
        else:
            self.entropy_threshold = 3.0

        if gaps:
            self.gap_threshold = min(gaps) * 0.3
        else:
            self.gap_threshold = 0.01

        print(f'OOD Calibration complete:')
        print(f'  Entropy threshold: {self.entropy_threshold:.4f} (reject if above)')
        print(f'  Gap threshold:     {self.gap_threshold:.4f} (reject if below)')
        print(f'  Calibrated on {len(entropies)} correct predictions')
        print(f'  Entropy range seen: [{min(entropies):.4f}, {max(entropies):.4f}]')
        print(f'  Gap range seen:     [{min(gaps):.4f}, {max(gaps):.4f}]')

    def predict(self, raw_input, top_k=3):
        """
        Predict with OOD rejection.
        Returns dict with 'rejected' flag.
        """
        text = raw_input.strip().lower()
        result = self.predictor.predict_with_confidence(text, top_k=top_k)

        # Compute rejection signals
        probs = [p for _, p in result['top_k']]
        ent = -sum(p * np.log(p + 1e-10) for p in probs if p > 0)
        gap = (probs[0] - probs[1]) if len(probs) >= 2 else probs[0]

        rejected = False
        reject_reason = None
        if self.entropy_threshold and ent > self.entropy_threshold:
            rejected = True
            reject_reason = f'high entropy ({ent:.3f} > {self.entropy_threshold:.3f})'
        elif self.gap_threshold and gap < self.gap_threshold:
            rejected = True
            reject_reason = f'low confidence gap ({gap:.3f} < {self.gap_threshold:.3f})'

        result['entropy']       = round(ent, 4)
        result['gap']           = round(gap, 4)
        result['rejected']      = rejected
        result['reject_reason'] = reject_reason
        return result


# ── Instantiate and Calibrate ────────────────────────────────────────────────
gated = ConfidenceGate(hier)
gated.calibrate(texts_train, y_train)

# ── Test with OOD inputs ─────────────────────────────────────────────────────
ood_inputs = [
    "Turn on bluetooth connectivity",
    "Send a text message to John",
    "Calculate body mass index for the user",
    "Check the weather forecast for today",
    "Order food from the restaurant app",
]

print(f'\n{"=" * 75}')
print(f'         OOD REJECTION TEST')
print(f'{"=" * 75}')
print(f'{"Input":<45} {"Prediction":<25} {"Rejected?":<10}')
print('─' * 75)

for inp in ood_inputs:
    r = gated.predict(inp)
    status = 'REJECTED' if r['rejected'] else 'ACCEPTED'
    print(f'{inp[:44]:<45} {r["prediction"]:<25} {status:<10}')

# Test in-distribution inputs (should NOT be rejected)
id_inputs = [
    "start my running workout and track calories",
    "stop the yoga session and save data",
    "show me my live heart rate readings",
    "get my sleep history data",
    "enable the music remote control",
]

print(f'\n{"In-Distribution Inputs":<45} {"Prediction":<25} {"Rejected?":<10}')
print('─' * 75)
for inp in id_inputs:
    r = gated.predict(inp)
    status = 'REJECTED' if r['rejected'] else 'ACCEPTED'
    print(f'{inp[:44]:<45} {r["prediction"]:<25} {status:<10}')

# Calculate false rejection rate on test set
false_rejects = 0
for text in texts_test:
    r = gated.predict(text)
    if r['rejected']:
        false_rejects += 1
fpr = false_rejects / len(texts_test)
print(f'\nFalse rejection rate (test set): {fpr:.2%} ({false_rejects}/{len(texts_test)})')

OOD Calibration complete:
  Entropy threshold: 1.7675 (reject if above)
  Gap threshold:     0.0746 (reject if below)
  Calibrated on 140 correct predictions
  Entropy range seen: [0.5517, 0.8837]
  Gap range seen:     [0.2488, 0.5529]

         OOD REJECTION TEST
Input                                         Prediction                Rejected? 
───────────────────────────────────────────────────────────────────────────
Turn on bluetooth connectivity                enableCounterControl      REJECTED  
Send a text message to John                   startTennis               REJECTED  
Calculate body mass index for the user        stress                    REJECTED  
Check the weather forecast for today          stopSwimming              REJECTED  
Order food from the restaurant app            stopSwimming              REJECTED  

In-Distribution Inputs                        Prediction                Rejected? 
───────────────────────────────────────────────────────────────────────────
s

## Section 6 — Production Inference Demo

The unified inference function that combines all layers: Hierarchical Router + Character N-gram Fusion + OOD Reject Gate.

In [9]:
# ── Production Inference Function ─────────────────────────────────────────────

def predict_function_name(raw_input, top_k=3):
    """
    Predict a camelCase function name from a natural-language command.
    Uses hierarchical router with OOD rejection.

    Args:
        raw_input (str): Natural language description, e.g. "start my yoga workout"
        top_k (int): Number of top candidates to return

    Returns:
        dict with: prediction, intent, confidence, top_k, rejected, latency_ms
    """
    return gated.predict(raw_input, top_k=top_k)


# ── Comprehensive Test Battery ───────────────────────────────────────────────
test_cases = [
    # In-distribution: one from each intent group
    {'label': 'Start workout',    'input': 'begin recording a swimming exercise session'},
    {'label': 'Stop workout',     'input': 'finish the basketball workout and save data'},
    {'label': 'Live monitor',     'input': 'stream my live heart rate from sensors'},
    {'label': 'Get data',         'input': 'retrieve my sleep history and summary'},
    {'label': 'Enable control',   'input': 'enable the ebook reading remote control'},
    {'label': 'Disable control',  'input': 'disable the short video remote feature'},

    # Paraphrased inputs (semantic variance test)
    {'label': 'Paraphrase 1',     'input': 'I want to start a yoga session'},
    {'label': 'Paraphrase 2',     'input': 'end my run and save it'},
    {'label': 'Paraphrase 3',     'input': 'what is my current blood oxygen level'},
    {'label': 'Paraphrase 4',     'input': 'show me my stress data'},

    # OOD inputs (should be rejected)
    {'label': 'OOD 1',           'input': 'play my favorite playlist on spotify'},
    {'label': 'OOD 2',           'input': 'set an alarm for 6 AM tomorrow'},
]

print('=' * 95)
print(f'{"Label":<18} {"Input":<42} {"Prediction":<24} {"Conf":>6} {"Status":<10}')
print('=' * 95)

for tc in test_cases:
    r = predict_function_name(tc['input'])
    conf = r['func_conf']
    status = 'REJECTED' if r['rejected'] else 'OK'
    print(f'{tc["label"]:<18} {tc["input"][:41]:<42} {r["prediction"]:<24} {conf:>6.2f} {status:<10}')
    if r['top_k']:
        top_str = ', '.join(f'{n}({p:.2f})' for n, p in r['top_k'][:3])
        print(f'{"":>18} Top-3: {top_str}')
    print()

Label              Input                                      Prediction                 Conf Status    
Start workout      begin recording a swimming exercise sessi  startSwimming              0.35 OK        
                   Top-3: startSwimming(0.35), startBadminton(0.03), startSkipping(0.03)

Stop workout       finish the basketball workout and save da  stopBasketball             0.32 OK        
                   Top-3: stopBasketball(0.32), stopVolleyball(0.03), stopCricket(0.03)

Live monitor       stream my live heart rate from sensors     liveHeartRate              0.45 OK        
                   Top-3: liveHeartRate(0.45), liveHrv(0.22), liveTemperature(0.17)

Get data           retrieve my sleep history and summary      sleep                      0.42 OK        
                   Top-3: sleep(0.42), activity(0.09), temperature(0.09)

Enable control     enable the ebook reading remote control    enableEbookReadingControl   0.49 OK        
                   Top-3: enabl

## Section 7 — Export & Deployment Verification

Export the full hierarchical pipeline as deployment artifacts. Verify by reloading from disk and running end-to-end inference.

In [10]:
# ── Export Artifacts ──────────────────────────────────────────────────────────

joblib.dump(hier,   'iot_predictor.pkl')
joblib.dump(le,     'iot_label_encoder.pkl')
joblib.dump({
    'entropy_threshold': gated.entropy_threshold,
    'gap_threshold':     gated.gap_threshold
}, 'iot_ood_config.pkl')

# Report file sizes
artifacts = ['iot_predictor.pkl', 'iot_label_encoder.pkl', 'iot_ood_config.pkl']
total_kb = 0
print(f'{"Artifact":<30} {"Size":>10}')
print('─' * 42)
for f in artifacts:
    kb = os.path.getsize(f) / 1024
    total_kb += kb
    print(f'{f:<30} {kb:>8.1f} KB')
print('─' * 42)
print(f'{"TOTAL":<30} {total_kb:>8.1f} KB')

limit_kb = 5 * 1024  # 5 MB
assert total_kb < limit_kb, f'Package too large: {total_kb:.1f} KB (limit: {limit_kb} KB)'
print(f'\nSize check PASSED — {total_kb:.1f} KB < {limit_kb} KB (5 MB embedded limit)')

# ── Reload & Verify ──────────────────────────────────────────────────────────
loaded_hier = joblib.load('iot_predictor.pkl')
loaded_le   = joblib.load('iot_label_encoder.pkl')
loaded_ood  = joblib.load('iot_ood_config.pkl')

# Reconstruct gated predictor from loaded artifacts
loaded_gated = ConfidenceGate(
    loaded_hier,
    entropy_threshold=loaded_ood['entropy_threshold'],
    gap_threshold=loaded_ood['gap_threshold']
)

# End-to-end verification
test_input = 'start my yoga workout and track calories'
result = loaded_gated.predict(test_input)

print(f'\n{"=" * 55}')
print(f'        DEPLOYMENT VERIFICATION')
print(f'{"=" * 55}')
print(f'Input      : {test_input}')
print(f'Prediction : {result["prediction"]}')
print(f'Intent     : {result["intent"]}')
print(f'Confidence : {result["func_conf"]}')
print(f'Rejected   : {result["rejected"]}')
print(f'Latency    : {result["latency_ms"]} ms')
print(f'{"=" * 55}')
print(f'Artifacts reloaded and verified successfully.')

Artifact                             Size
──────────────────────────────────────────
iot_predictor.pkl                 856.3 KB
iot_label_encoder.pkl               1.9 KB
iot_ood_config.pkl                  0.2 KB
──────────────────────────────────────────
TOTAL                             858.4 KB

Size check PASSED — 858.4 KB < 5120 KB (5 MB embedded limit)



        DEPLOYMENT VERIFICATION
Input      : start my yoga workout and track calories
Prediction : startYoga
Intent     : start_workout
Confidence : 0.2309
Rejected   : False
Latency    : 1.0979 ms
Artifacts reloaded and verified successfully.


In [11]:
# ── Final Comparison Summary ─────────────────────────────────────────────────

print('=' * 80)
print('                     MODEL COMPARISON SUMMARY')
print('=' * 80)
print(f'{"Metric":<30} {"LR(word)":<15} {"LR(word+char)":<15} {"Hierarchical":<15}')
print('─' * 80)

r_w  = flat_results.get('LR (word only)', {})
r_wc = flat_results.get('LR (word+char)', {})

def fmt(v, f='.4f'):
    return f'{v:{f}}' if v is not None else 'N/A'

print(f'{"Test Accuracy":<30} {fmt(r_w.get("test_acc")):<15} {fmt(r_wc.get("test_acc")):<15} {fmt(hier_acc):<15}')
print(f'{"CV Accuracy (mean)":<30} {fmt(r_w.get("cv_mean")):<15} {fmt(r_wc.get("cv_mean")):<15} {"—":<15}')
print(f'{"Top-3 Accuracy":<30} {fmt(r_w.get("top3_acc")):<15} {fmt(r_wc.get("top3_acc")):<15} {fmt(hier_top3):<15}')
print(f'{"Inference Latency (ms)":<30} {fmt(r_w.get("infer_ms")):<15} {fmt(r_wc.get("infer_ms")):<15} {fmt(lat.mean()):<15}')
print(f'{"Model Size (KB)":<30} {"—":<15} {"—":<15} {fmt(total_kb, ".1f"):<15}')
print(f'{"OOD Rejection (OOD inputs)":<30} {"—":<15} {"—":<15} {"Active":<15}')
print(f'{"False Reject Rate":<30} {"—":<15} {"—":<15} {fmt(fpr, ".2%"):<15}')
print('=' * 80)

print(f'\n{"=" * 55}')
print(f'            PROJECT SUMMARY')
print(f'{"=" * 55}')
print(f'Dataset          : IoT Wearable SDK (improved_functions_1.csv)')
print(f'Total rows       : {len(df)} (augmented from {df_raw.shape[0]})')
print(f'Unique functions : {df["function_name"].nunique()}')
print(f'Intent groups    : {df["intent"].nunique()}')
print(f'Best model       : Hierarchical Router (2-stage)')
print(f'Test accuracy    : {hier_acc:.4f}')
print(f'Top-3 accuracy   : {hier_top3:.4f}')
print(f'Model pkg size   : {total_kb:.1f} KB')
print(f'OOD detection    : Active (entropy + gap thresholding)')
print(f'{"=" * 55}')

                     MODEL COMPARISON SUMMARY
Metric                         LR(word)        LR(word+char)   Hierarchical   
────────────────────────────────────────────────────────────────────────────────
Test Accuracy                  0.7000          0.9857          1.0000         
CV Accuracy (mean)             0.4371          0.7714          —              
Top-3 Accuracy                 0.9000          1.0000          1.0000         
Inference Latency (ms)         0.0794          0.1063          0.9701         
Model Size (KB)                —               —               858.4          
OOD Rejection (OOD inputs)     —               —               Active         
False Reject Rate              —               —               0.00%          

            PROJECT SUMMARY
Dataset          : IoT Wearable SDK (improved_functions_1.csv)
Total rows       : 210 (augmented from 70)
Unique functions : 70
Intent groups    : 6
Best model       : Hierarchical Router (2-stage)
Test accuracy 